# Creating the chat agent with AutoGen

In [17]:
# Importing environment variables
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Importing autogen agents, messages and models

In [18]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
openai_model = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [19]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollam_client = OllamaChatCompletionClient(model="llama3.2")

In [20]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="let's go to Japan", source="user")

from autogen_agentchat.agents import AssistantAgent
openai_agent = AssistantAgent(
    name="Airline_Agent",
    model_client=openai_model,
    system_message="You are helpful a very helpful assistant for an Airline. You give short and humorous answer.",
    model_client_stream=True
)

### Combining all to complete the chat communication

In [21]:
from autogen_core import CancellationToken

response = await openai_agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'Absolutely! Just don’t forget to pack your sushi-eating chopsticks and a good pair of walking shoes for all that sightseeing… and maybe a guidebook on how to politely say, "Where\'s the karaoke bar?"! 🍣🎤✈️'

In [22]:
### using Ollama client
ollama_agent = AssistantAgent(
    model_client=ollam_client,
    name="ollama_agent",
    system_message="You are helpful a very helpful assistant for an Airline. You give short and humorous answer.",
    model_client_stream=True
)

response = await ollama_agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

"Japan-bound? That's a NO-FLY-ZONE for snacks and anime! Book those tickets, and don't forget to pack your chopstick skills"

## Creating a local database with sqlite to store ticket prices

In [24]:
import os
import sqlite3

from annotated_types import LowerCase

# Remove any exisiting database
if os.path.exists("flight_prices.db"):
    os.remove("flight_prices.db")

# Create database connection
conn = sqlite3.connect("flight_prices.db")
c = conn.cursor()
c.execute("CREATE TABLE Cities(city_name TEXT PRIMARY_KEY, round_trip_price REAL)")
conn.commit()
conn.close()

## function to insert values to the table
def add_ticket_prices(city_name, price):
    conn = sqlite3.connect("flight_prices.db")
    c = conn.cursor()
    c.execute("REPLACE INTO Cities(city_name, round_trip_price) values(?,?)",(city_name.lower(), price))
    conn.commit()
    conn.close()

In [25]:
## Adding city prices
add_ticket_prices("Landon", 788.00)
add_ticket_prices("Paris", 715.20)
add_ticket_prices("Tokyo", 899.00)
add_ticket_prices("Madrid", 677.00)
add_ticket_prices("Barlin", 600.00)
add_ticket_prices("Rome", 560.00)

In [26]:
## function to get ticket price
def get_ticket_price(city: str) -> float:
    conn= sqlite3.connect("flight_prices.db")
    c=conn.cursor()
    c.execute("SELECT round_trip_price from Cities where city_name =?", (city.lower(),))
    result = c.fetchone()
    conn.close()
    return result

In [27]:
get_ticket_price("Tokyo")

(899.0,)

In [41]:
smart_agent = AssistantAgent(
    name="Smart_Agent",
    model_client=openai_model,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    tools=[get_ticket_price],
    model_client_stream=True,
    reflect_on_tool_use=True
)

trip_info = TextMessage( content="I would like to go to Paris", source="user")

response = await smart_agent.on_messages([trip_info], cancellation_token=CancellationToken())
for resp in response.inner_messages:
    print(resp.content)
response.chat_message.content

[FunctionCall(id='call_2mrhAf0viO5RR4jYcwsZi8ec', arguments='{"city":"Paris"}', name='get_ticket_price')]
[FunctionExecutionResult(content='(715.2,)', name='get_ticket_price', call_id='call_2mrhAf0viO5RR4jYcwsZi8ec', is_error=False)]


'Ah, Paris! The city of love... and overpriced coffee! A roundtrip ticket will cost you about $715.20. Bon voyage!'

## Now using the tool call for Ollama model

In [42]:
smart_agent = AssistantAgent(
    name="Smart_Ollama_Agent",
    model_client=ollam_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    tools=[get_ticket_price],
    model_client_stream=True,
    reflect_on_tool_use=True
)

trip_info = TextMessage( content="I would like to go to Rome", source="user")

response = await smart_agent.on_messages([trip_info], cancellation_token=CancellationToken())
for resp in response.inner_messages:
    print(resp.content)
response.chat_message.content

[FunctionCall(id='0', arguments='{"city": "Rome"}', name='get_ticket_price')]
[FunctionExecutionResult(content='(560.0,)', name='get_ticket_price', call_id='0', is_error=False)]


"You want to fly to Rome, huh? Well, buckle up, because you're going to need some serious pasta-fueled energy to navigate all the ancient history. Roundtrip from the US, the ticket will set you back about $560. Worth it, right?"